# FCITEM Monthly Demand Forecasting — Option A Production AWS MLOps Pipeline v4 (Direct Local Tests + CloudShell Docker Build)
**End-to-End Production Architecture (Option A):**
1. **Hardened AWS Glue Python Shell (`glue_preprocess.py`):** Auto-selects the single latest `FCITEM` file in `s3://clariant-aiml-dev/raw/`, ignores `~$` Excel lock files, filters dead/sparse SKUs, computes leak-free causal features + SBC demand classes, and writes `processed/clean.parquet` + `processed/dq_report.json`.
2. **Batch Training + Drift Gate (`pipeline.py` on ECS Fargate):** Evaluates **XGBoost / ETS / Croston's SBA** on a 3-month holdout, writes `forecast.csv` & `forecast_wide.csv` to S3, emits metrics to **CloudWatch (`FCITEM/Forecasting`)**, and sends **SNS** alerts.
3. **SageMaker Model Registry + Real-Time Endpoint (`ml.m5.large`) + Inbuilt Model Monitor:** Registers versioned Model Packages (`fcitem-forecast-group`), auto-approves by WAPE threshold, deploys `fcitem-forecast-endpoint` with **100% Data Capture**, and schedules **SageMaker Model Monitor** Data Quality checks.
4. **Live BI & App REST API (`api.py` on ECS Fargate Service):** FastAPI + Uvicorn running on port `8080` from the same ECR image (`/health`, `/forecast/{fcitem}`, `/forecast/batch`, `/portfolio/summary`, `/model-info`) for Power BI / Tableau / QuickSight.
5. **CloudWatch Observability Dashboard (`FCITEM-MLOps-Dashboard`):** Unified dashboard tracking WAPE, PSI drift, Champion model mix, Glue DQ, ECS Live API health, and SageMaker Endpoint metrics.

In [ ]:
import os, sys, io, json, time, shutil, zipfile, subprocess, boto3, numpy as np, pandas as pd
from botocore.exceptions import ClientError

# ==============================================================================
# 1. EDIT THESE CONFIG VALUES BEFORE RUNNING
# ==============================================================================
REGION      = "eu-north-1"
BUCKET      = "clariant-aiml-dev"
ALERT_EMAIL = "you@example.com"          # <-- Replace with your email & confirm SNS subscription

ITEM_COL    = "FCITEM"                   # ID column in your wide monthly file
RAW_PREFIX  = "raw/"
HORIZON     = 18                         # 18-month forecast horizon
SM_MODEL_GROUP   = "fcitem-forecast-group"
SM_ENDPOINT_NAME = "fcitem-forecast-endpoint"
SM_INSTANCE_TYPE = "ml.m5.large"
# ==============================================================================

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PIPELINE_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/forecast-pipeline-role"
ECR_REPO = "forecast-pipeline"
ECR_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:latest"

CLUSTER, TASK_FAMILY, LIVE_SERVICE, CONTAINER = (
    "forecast-cluster", "forecast-task", "forecast-live-service", "pipeline"
)
GLUE_JOB, SFN_NAME, TOPIC_NAME, LOG_GROUP = (
    "forecast-preprocess", "forecast-monthly-pipeline", "forecast-alerts", "/ecs/forecast-task"
)
DASHBOARD_NAME = "FCITEM-MLOps-Dashboard"

s3 = boto3.client("s3", region_name=REGION)
ecr = boto3.client("ecr", region_name=REGION)
ecs = boto3.client("ecs", region_name=REGION)
glue = boto3.client("glue", region_name=REGION)
sns = boto3.client("sns", region_name=REGION)
sfn = boto3.client("stepfunctions", region_name=REGION)
events = boto3.client("events", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
ec2 = boto3.client("ec2", region_name=REGION)
cw = boto3.client("cloudwatch", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)

vpc_id = ec2.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])["Vpcs"][0]["VpcId"]
subnets = [x["SubnetId"] for x in ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpc_id]}])["Subnets"]]
sg_id = ec2.describe_security_groups(
    Filters=[{"Name": "vpc-id", "Values": [vpc_id]}, {"Name": "group-name", "Values": ["default"]}]
)["SecurityGroups"][0]["GroupId"]

# Ensure TCP 8080 is open on default SG so BI tools / browsers can reach the ECS Live API
try:
    ec2.authorize_security_group_ingress(
        GroupId=sg_id, IpProtocol="tcp", FromPort=8080, ToPort=8080, CidrIp="0.0.0.0/0"
    )
    print("Opened inbound TCP 8080 on Security Group:", sg_id)
except Exception:
    pass

os.makedirs("app/tests", exist_ok=True)
print(f"Account: {ACCOUNT_ID} | Region: {REGION}")
print(f"Subnets: {subnets} | Security Group: {sg_id}")

## A. Hardened AWS Glue Python Shell (`raw/` -> `processed/clean.parquet` + `processed/dq_report.json`)
- **Latest-File Selection:** Automatically picks only the single most recently uploaded file in `raw/` (preventing double-counting when new monthly dumps arrive) and skips Excel lock files (`~$`).
- **Smart Excel Sheet Detection:** Finds the worksheet containing `FCITEM` automatically.
- **Production SKU Quality Gates:** Filters `< 15` months history, `< 4` positive demand months, and dead SKUs with `0` sales in the trailing 12 months.
- **Idempotent Job Reconciliation:** Uses `glue.get_job()` first (avoiding `IdempotentParameterMismatchException` on `create_job`), stops any stale in-flight Glue runs, and automatically falls back to `delete_job` + `create_job` if immutable job parameters changed.

In [ ]:
%%writefile glue_preprocess.py
import sys, io, re, json, datetime as dt, boto3, numpy as np, pandas as pd
from awsglue.utils import getResolvedOptions

args = getResolvedOptions(
    sys.argv,
    ["S3_BUCKET", "RAW_PREFIX", "PROCESSED_KEY", "DQ_KEY", "ITEM_COL"]
)

MIN_HISTORY = 15          # 12 train months + 3 holdout months
MIN_POSITIVE_MONTHS = 4   # Require at least 4 non-zero demand months
MAX_TRAILING_ZEROS = 12   # Skip SKUs with 0 demand across the last 12 months
LAGS = [1, 2, 3, 6, 12]


def parse_month_col(col):
    """Converts wide monthly column headers (Timestamp, YYYY-MM, Oct-22, YYYYMM, etc.) to Month-Start Timestamp."""
    if isinstance(col, (pd.Timestamp, dt.datetime, dt.date, np.datetime64)):
        ts = pd.to_datetime(col, errors="coerce")
        return pd.NaT if pd.isna(ts) else ts.to_period("M").to_timestamp()

    s = str(col).strip()
    if not s or s.lower() in ("total", "grand total", "sum", "average", "avg", "unnamed: 0"):
        return pd.NaT

    explicit_formats = [
        "%Y%m", "%Y-%m", "%Y/%m", "%b-%y", "%b %y",
        "%b-%Y", "%b %Y", "%y-%b", "%Y-%b", "%m-%Y", "%m/%Y"
    ]
    for fmt in explicit_formats:
        try:
            ts = pd.to_datetime(s, format=fmt)
            return ts.to_period("M").to_timestamp()
        except Exception:
            pass

    ts = pd.to_datetime(s, errors="coerce")
    if pd.isna(ts):
        return pd.NaT
    return ts.to_period("M").to_timestamp()


def read_excel_smart(body_bytes, item_col_hint="FCITEM"):
    """Reads the worksheet containing ITEM_COL (or falls back to the first non-empty sheet)."""
    xl = pd.ExcelFile(io.BytesIO(body_bytes))
    target_sheet = xl.sheet_names[0]
    for sheet in xl.sheet_names:
        preview = xl.parse(sheet, nrows=5)
        cols_upper = [str(c).strip().upper() for c in preview.columns]
        if item_col_hint.strip().upper() in cols_upper:
            target_sheet = sheet
            break
    return xl.parse(target_sheet), target_sheet


def unpivot_fcitem_wide(df, item_col_hint="FCITEM"):
    """Unpivots wide table [FCITEM, month_1, month_2, ...] into long [item, date, y]."""
    cols = list(df.columns)
    item_col = None
    for c in cols:
        if str(c).strip().upper() == item_col_hint.strip().upper():
            item_col = c
            break
    if item_col is None:
        item_col = cols[0]

    month_col_map = {}
    for c in cols:
        if c == item_col:
            continue
        dt_val = parse_month_col(c)
        if not pd.isna(dt_val):
            month_col_map[c] = dt_val

    if not month_col_map:
        raise ValueError(f"No monthly date columns detected alongside '{item_col}'. Columns found: {cols[:10]}")

    sub = df[[item_col] + list(month_col_map.keys())].copy()
    sub = sub.rename(columns={item_col: "item"})
    long_df = sub.melt(id_vars=["item"], var_name="raw_col", value_name="y")
    long_df["date"] = long_df["raw_col"].map(month_col_map)
    long_df = long_df.drop(columns=["raw_col"])
    return long_df


def classify_sbc(y):
    """Syntetos-Boylan Classification (SBC) using ADI and squared CV."""
    y = np.asarray(y, float)
    nz = y[y > 0]
    if len(nz) < 2:
        return "intermittent"
    adi = len(y) / len(nz)
    cv2 = (np.std(nz) / max(np.mean(nz), 1e-6)) ** 2
    if adi < 1.32 and cv2 < 0.49:
        return "smooth"
    elif adi < 1.32 and cv2 >= 0.49:
        return "erratic"
    elif adi >= 1.32 and cv2 < 0.49:
        return "intermittent"
    return "lumpy"


def engineer_causal_features(df):
    """Computes leak-free causal lags, rolling stats, intermittency signals, SBC class, and calendar features."""
    df = df.sort_values(["item", "date"]).reset_index(drop=True)

    sbc_rows = []
    for it, g in df.groupby("item"):
        sbc_rows.append({"item": it, "demand_class": classify_sbc(g["y"].values)})
    df = df.merge(pd.DataFrame(sbc_rows), on="item", how="left")

    # Causal expanding mean (up to t-1) to prevent future target leakage in clean.parquet
    g_raw = df.groupby("item")["y"]
    shifted_y = g_raw.shift(1)
    exp_mean = shifted_y.groupby(df["item"]).transform(
        lambda s: s.expanding(min_periods=1).mean()
    ).fillna(df["y"]).clip(lower=1e-6)
    df["ys"] = df["y"] / exp_mean

    g = df.groupby("item")["ys"]
    for l in LAGS:
        df[f"lag{l}"] = g.shift(l)

    shifted = g.shift(1)
    df["roll3_mean"] = shifted.groupby(df["item"]).transform(lambda s: s.rolling(3, min_periods=1).mean())
    df["roll6_mean"] = shifted.groupby(df["item"]).transform(lambda s: s.rolling(6, min_periods=1).mean())
    df["roll6_std"] = shifted.groupby(df["item"]).transform(lambda s: s.rolling(6, min_periods=2).std()).fillna(0.0)
    df["ewm3"] = shifted.groupby(df["item"]).transform(lambda s: s.ewm(span=3, adjust=False).mean())

    is_zero = (shifted == 0).astype(float)
    df["zero_ratio6"] = is_zero.groupby(df["item"]).transform(lambda s: s.rolling(6, min_periods=1).mean()).fillna(0.0)

    def _months_since_nz(arr):
        out = np.zeros(len(arr), dtype=float)
        cnt = 0.0
        for i, v in enumerate(arr):
            out[i] = cnt
            if np.isnan(v) or v > 0:
                cnt = 0.0
            else:
                cnt += 1.0
        return out

    df["months_since_nz"] = shifted.groupby(df["item"]).transform(
        lambda s: pd.Series(_months_since_nz(s.values), index=s.index)
    )

    m = df["date"].dt.month
    df["month_sin"] = np.sin(2.0 * np.pi * m / 12.0)
    df["month_cos"] = np.cos(2.0 * np.pi * m / 12.0)
    df["quarter"] = df["date"].dt.quarter
    return df


def select_latest_raw_object(s3_client, bucket, prefix):
    """Selects ONLY the single most recently uploaded valid raw file (ignores ~$ temp/lock files)."""
    candidates = []
    for page in s3_client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for o in page.get("Contents", []):
            key = o["Key"]
            fname = key.split("/")[-1]
            low = fname.lower()
            if not fname or fname.startswith("~$") or fname.startswith("."):
                continue
            if not low.endswith((".parquet", ".xlsx", ".xlsm", ".csv")):
                continue
            candidates.append(o)
    if not candidates:
        raise RuntimeError(f"No valid raw files found under s3://{bucket}/{prefix}")
    candidates.sort(key=lambda x: x["LastModified"], reverse=True)
    return candidates[0]


def publish_glue_dq_metrics(cw_client, dq):
    try:
        cw_client.put_metric_data(
            Namespace="FCITEM/Forecasting",
            MetricData=[
                {"MetricName": "GlueInputRows", "Value": float(dq["raw_unpivoted_rows"]), "Unit": "Count"},
                {"MetricName": "GlueValidItems", "Value": float(dq["valid_items"]), "Unit": "Count"},
                {"MetricName": "GlueExcludedItems", "Value": float(dq["excluded_items_total"]), "Unit": "Count"},
            ],
        )
    except Exception as e:
        print(f"Warning: CloudWatch DQ metric publish failed: {e}")


def run():
    s3 = boto3.client("s3")
    cw = boto3.client("cloudwatch")

    latest_obj = select_latest_raw_object(s3, args["S3_BUCKET"], args["RAW_PREFIX"])
    key = latest_obj["Key"]
    low = key.lower()
    raw_bytes = s3.get_object(Bucket=args["S3_BUCKET"], Key=key)["Body"].read()

    sheet_used = "N/A"
    if low.endswith(".parquet"):
        d = pd.read_parquet(io.BytesIO(raw_bytes))
    elif low.endswith(".csv"):
        d = pd.read_csv(io.BytesIO(raw_bytes))
    else:
        d, sheet_used = read_excel_smart(raw_bytes, item_col_hint=args["ITEM_COL"])

    long_df = unpivot_fcitem_wide(d, item_col_hint=args["ITEM_COL"])
    raw_rows = len(long_df)

    long_df["item"] = long_df["item"].astype(str).str.strip()
    long_df = long_df[long_df["item"].notna() & (long_df["item"] != "") & (long_df["item"].str.lower() != "nan")]
    numeric_y = pd.to_numeric(long_df["y"], errors="coerce").fillna(0.0)
    neg_clipped = int((numeric_y < 0).sum())
    long_df["y"] = numeric_y.clip(lower=0.0)
    long_df = long_df.dropna(subset=["item", "date"])

    pre_dedup_rows = len(long_df)
    long_df = long_df.groupby(["item", "date"], as_index=False)["y"].sum()
    duplicates_merged = int(pre_dedup_rows - len(long_df))

    end = long_df["date"].max()
    parts = []
    excl_short = 0
    excl_sparse = 0
    excl_inactive = 0

    for it, g in long_df.groupby("item"):
        idx = pd.date_range(g["date"].min(), end, freq="MS")
        s = g.set_index("date")["y"].reindex(idx, fill_value=0.0)
        vals = s.values
        if len(vals) < MIN_HISTORY:
            excl_short += 1
            continue
        if int((vals > 0).sum()) < MIN_POSITIVE_MONTHS:
            excl_sparse += 1
            continue
        if float(vals[-MAX_TRAILING_ZEROS:].sum()) <= 0.0:
            excl_inactive += 1
            continue
        parts.append(pd.DataFrame({"item": it, "date": idx, "y": vals}))

    if not parts:
        raise RuntimeError(f"No FCITEM series met quality gates (MIN_HISTORY={MIN_HISTORY}, MIN_POS={MIN_POSITIVE_MONTHS}).")

    panel = pd.concat(parts, ignore_index=True)
    out = engineer_causal_features(panel)

    buf = io.BytesIO()
    out.to_parquet(buf, index=False)
    s3.put_object(Bucket=args["S3_BUCKET"], Key=args["PROCESSED_KEY"], Body=buf.getvalue())

    sbc_counts = out.groupby("item")["demand_class"].first().value_counts().to_dict()
    dq_report = {
        "processed_at_utc": dt.datetime.utcnow().isoformat() + "Z",
        "source_file": f"s3://{args['S3_BUCKET']}/{key}",
        "source_last_modified": str(latest_obj["LastModified"]),
        "excel_sheet_used": sheet_used,
        "raw_wide_rows": int(d.shape[0]),
        "raw_unpivoted_rows": int(raw_rows),
        "negatives_clipped_to_zero": neg_clipped,
        "duplicate_item_months_merged": duplicates_merged,
        "valid_items": int(out["item"].nunique()),
        "excluded_short_history": excl_short,
        "excluded_sparse_lt4_pos": excl_sparse,
        "excluded_inactive_12m": excl_inactive,
        "excluded_items_total": int(excl_short + excl_sparse + excl_inactive),
        "output_rows": int(len(out)),
        "date_min": str(out["date"].min().date()),
        "date_max": str(out["date"].max().date()),
        "sbc_breakdown": {str(k): int(v) for k, v in sbc_counts.items()},
    }
    s3.put_object(
        Bucket=args["S3_BUCKET"],
        Key=args["DQ_KEY"],
        Body=json.dumps(dq_report, indent=2).encode("utf-8"),
    )
    publish_glue_dq_metrics(cw, dq_report)
    print("DQ REPORT:", json.dumps(dq_report))


if __name__ == "__main__":
    run()

In [ ]:
s3.upload_file("glue_preprocess.py", BUCKET, "scripts/glue_preprocess.py")

GLUE_DEFAULT_ARGS = {
    "--library-set": "analytics",
    "--additional-python-modules": "openpyxl,pyarrow",
    "--S3_BUCKET": BUCKET,
    "--RAW_PREFIX": RAW_PREFIX,
    "--PROCESSED_KEY": "processed/clean.parquet",
    "--DQ_KEY": "processed/dq_report.json",
    "--ITEM_COL": ITEM_COL,
}

job = dict(
    Role=PIPELINE_ROLE_ARN,
    Command={
        "Name": "pythonshell",
        "ScriptLocation": f"s3://{BUCKET}/scripts/glue_preprocess.py",
        "PythonVersion": "3.9",
    },
    DefaultArguments=GLUE_DEFAULT_ARGS,
    ExecutionProperty={"MaxConcurrentRuns": 3},
    GlueVersion="3.0",
    MaxCapacity=1.0,
    Timeout=60,
    MaxRetries=0,
)

# Check if Glue job already exists via get_job() so create_job never hits IdempotentParameterMismatchException
job_exists = False
try:
    glue.get_job(JobName=GLUE_JOB)
    job_exists = True
except ClientError as e:
    if e.response["Error"]["Code"] != "EntityNotFoundException":
        raise

if job_exists:
    # Stop any active/queued Glue runs from an earlier configuration
    runs = glue.get_job_runs(JobName=GLUE_JOB, MaxResults=10).get("JobRuns", [])
    active_ids = [r["Id"] for r in runs if r["JobRunState"] in ("STARTING", "RUNNING", "STOPPING", "WAITING")]
    if active_ids:
        glue.batch_stop_job_run(JobName=GLUE_JOB, JobRunIds=active_ids)
        print(f"Stopped {len(active_ids)} in-flight Glue run(s):", active_ids)
        time.sleep(5)

    try:
        glue.update_job(JobName=GLUE_JOB, JobUpdate=job)
        print("Glue job updated:", GLUE_JOB)
    except ClientError as e:
        print(f"Immutable Glue setting changed ({e.response['Error']['Code']}) — recreating {GLUE_JOB} cleanly...")
        glue.delete_job(JobName=GLUE_JOB)
        time.sleep(2)
        glue.create_job(Name=GLUE_JOB, **job)
        print("Glue job recreated:", GLUE_JOB)
else:
    glue.create_job(Name=GLUE_JOB, **job)
    print("Glue job created:", GLUE_JOB)

## B. Unified Container Application (`pipeline.py` + `api.py` + `sagemaker_handler.py` + `tests/`)
One Docker image powers **three** workloads:
1. **Batch Training & Drift Gate (`pipeline.py`):** Evaluates XGBoost / ETS / Croston's SBA, registers versioned Model Packages in **SageMaker Model Registry**, deploys/updates the **SageMaker Real-Time Endpoint (`ml.m5.large`)** with Data Capture, and publishes **CloudWatch** custom metrics.
2. **SageMaker Inference Handler (`sagemaker_handler.py`):** Bundled inside `model.tar.gz` as `code/inference.py` to serve live JSON predictions on the SageMaker endpoint.
3. **ECS Live BI REST API (`api.py`):** FastAPI service on port `8080` for Power BI / Tableau / QuickSight (`/health`, `/model-info`, `/portfolio/summary`, `/forecast/{fcitem}`, `/forecast/batch`).

In [ ]:
%%writefile app/sagemaker_handler.py
import os
import io
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

LAGS = [1, 2, 3, 6, 12]
ENGINEERED_COLS = (
    [f"lag{l}" for l in LAGS]
    + ["roll3_mean", "roll6_mean", "roll6_std", "ewm3", "zero_ratio6",
       "months_since_nz", "month_sin", "month_cos", "quarter", "item_cv"]
)
FEATS = ENGINEERED_COLS + ["code"]


def croston_sba_fc(y, h, alpha=0.15):
    y = np.asarray(y, float)
    nz_idx = np.flatnonzero(y > 0)
    if len(nz_idx) == 0:
        return np.zeros(h, dtype=float)
    if len(nz_idx) == 1:
        return np.full(h, y[nz_idx[0]] / max(len(y), 1), dtype=float)
    z = y[nz_idx[0]]
    p = float(nz_idx[0] + 1)
    q = 1
    for t in range(nz_idx[0] + 1, len(y)):
        if y[t] > 0:
            z = alpha * y[t] + (1.0 - alpha) * z
            p = alpha * q + (1.0 - alpha) * p
            q = 1
        else:
            q += 1
    rate = (1.0 - alpha / 2.0) * (z / max(p, 1.0))
    return np.full(h, max(rate, 0.0), dtype=float)


def ets_fc(y, h):
    y = np.asarray(y, float)
    try:
        if len(y) >= 24 and np.std(y) > 1e-6:
            res = ETSModel(y, error="add", trend="add", damped_trend=True, seasonal="add", seasonal_periods=12).fit(disp=False)
        elif len(y) >= 6 and np.std(y) > 1e-6:
            res = ETSModel(y, error="add", trend="add", damped_trend=True).fit(disp=False)
        else:
            return np.repeat(np.mean(y[-3:]), h)
        f = np.asarray(res.forecast(h), float)
        if not np.isfinite(f).all():
            raise ValueError("non-finite ETS")
    except Exception:
        f = np.repeat(np.mean(y[-3:]), h)
    return np.clip(f, 0.0, None)


def _compute_features(d):
    d = d.copy()
    g = d.groupby("item")["ys"]
    for l in LAGS:
        d[f"lag{l}"] = g.shift(l)
    shifted = g.shift(1)
    d["roll3_mean"] = shifted.groupby(d["item"]).transform(lambda s: s.rolling(3, min_periods=1).mean())
    d["roll6_mean"] = shifted.groupby(d["item"]).transform(lambda s: s.rolling(6, min_periods=1).mean())
    d["roll6_std"] = shifted.groupby(d["item"]).transform(lambda s: s.rolling(6, min_periods=2).std()).fillna(0.0)
    d["ewm3"] = shifted.groupby(d["item"]).transform(lambda s: s.ewm(span=3, adjust=False).mean())
    is_zero = (shifted == 0).astype(float)
    d["zero_ratio6"] = is_zero.groupby(d["item"]).transform(lambda s: s.rolling(6, min_periods=1).mean()).fillna(0.0)

    def _months_since_nz(arr):
        out = np.zeros(len(arr), dtype=float)
        cnt = 0.0
        for i, v in enumerate(arr):
            out[i] = cnt
            cnt = 0.0 if (np.isnan(v) or v > 0) else cnt + 1.0
        return out

    d["months_since_nz"] = shifted.groupby(d["item"]).transform(
        lambda s: pd.Series(_months_since_nz(s.values), index=s.index)
    )
    m = d["date"].dt.month
    d["month_sin"] = np.sin(2.0 * np.pi * m / 12.0)
    d["month_cos"] = np.cos(2.0 * np.pi * m / 12.0)
    d["quarter"] = d["date"].dt.quarter
    return d


def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "bundle.joblib"))


def input_fn(request_body, request_content_type="application/json"):
    if request_content_type == "application/json":
        return json.loads(request_body)
    raise ValueError(f"Unsupported content type: {request_content_type}")


def predict_fn(input_data, bundle):
    """
    Expects JSON payload:
      {"fcitems": ["ITEM_A", "ITEM_B"], "horizon": 18}
    Returns champion model predictions per FCITEM using the trained bundle.
    """
    items = input_data.get("fcitems") or list(bundle.get("history", {}).keys())[:10]
    h = int(input_data.get("horizon", 18))
    xgb_model = bundle["xgb"]
    best_map = bundle.get("best", {})
    codes = bundle.get("codes", {})
    scales = bundle.get("scales", {})
    cv_map = bundle.get("cv_map", {})
    history = bundle.get("history", {})
    last_date = pd.to_datetime(bundle.get("last_date", "2025-12-01"))

    results = []
    for it in items:
        y_hist = np.asarray(history.get(it, []), dtype=float)
        if len(y_hist) == 0:
            results.append({"FCITEM": it, "error": "FCITEM not found in trained bundle"})
            continue

        champion = best_map.get(it, "croston" if np.mean(y_hist == 0) > 0.30 else "ets")
        if champion == "xgb" and it in codes:
            dates = pd.date_range(end=last_date, periods=len(y_hist), freq="MS")
            scale = max(float(scales.get(it, np.mean(y_hist))), 1e-6)
            base_hist = pd.DataFrame({
                "item": it,
                "date": dates,
                "ys": y_hist / scale,
                "code": int(codes[it]),
                "item_cv": float(cv_map.get(it, 0.0)),
            })
            for _ in range(h):
                nxt_date = base_hist["date"].max() + pd.DateOffset(months=1)
                nxt = pd.DataFrame([{
                    "item": it, "date": nxt_date, "ys": np.nan,
                    "code": int(codes[it]), "item_cv": float(cv_map.get(it, 0.0))
                }])
                base_hist = pd.concat([base_hist, nxt], ignore_index=True)
                feat_df = _compute_features(base_hist)
                sel = feat_df["ys"].isna()
                pred = float(np.clip(xgb_model.predict(feat_df.loc[sel, FEATS])[0], 0.0, None))
                base_hist.loc[sel, "ys"] = pred
            preds = base_hist["ys"].values[-h:] * scale
        elif champion == "croston":
            preds = croston_sba_fc(y_hist, h)
        else:
            preds = ets_fc(y_hist, h)

        fc_dates = pd.date_range(last_date + pd.DateOffset(months=1), periods=h, freq="MS")
        results.append({
            "FCITEM": it,
            "model": champion,
            "horizon": h,
            "forecasts": [
                {"date": d.strftime("%Y-%m-%d"), "forecast": round(float(v), 3)}
                for d, v in zip(fc_dates, preds)
            ],
        })
    return {"trained_at": bundle.get("trained_at"), "predictions": results}


def output_fn(prediction, response_content_type="application/json"):
    return json.dumps(prediction)

In [ ]:
%%writefile app/pipeline.py
import os, io, json, tarfile, tempfile, warnings, datetime as dt
import numpy as np, pandas as pd, boto3, joblib
import xgboost as xgb
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

warnings.filterwarnings("ignore")

BUCKET = os.environ.get("S3_BUCKET", "")
REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "eu-north-1"))
ROOT = os.environ.get("ROOT", "").strip("/")          # "" in prod, "demo" for live demo
MODE = os.environ.get("MODE", "monthly")              # monthly = always retrain | drift = retrain only if drift
HORIZON = int(os.environ.get("HORIZON", "18"))
HOLDOUT = int(os.environ.get("HOLDOUT", "3"))
MIN_TRAIN = int(os.environ.get("MIN_TRAIN", "12"))
PSI_THRESHOLD = float(os.environ.get("PSI_THRESHOLD", "0.25"))
ERR_THRESHOLD = float(os.environ.get("ERR_THRESHOLD", "0.50"))
APPROVAL_WAPE_THRESHOLD = float(os.environ.get("APPROVAL_WAPE_THRESHOLD", "0.50"))
SNS_TOPIC_ARN = os.environ.get("SNS_TOPIC_ARN", "")

# SageMaker Option A settings
ENABLE_SAGEMAKER = os.environ.get("ENABLE_SAGEMAKER", "true").lower() == "true"
SM_MODEL_GROUP = os.environ.get("SM_MODEL_GROUP", "fcitem-forecast-group")
SM_ENDPOINT_NAME = os.environ.get("SM_ENDPOINT_NAME", "fcitem-forecast-endpoint")
SM_INSTANCE_TYPE = os.environ.get("SM_INSTANCE_TYPE", "ml.m5.large")
SM_ROLE_ARN = os.environ.get("SM_ROLE_ARN", "")
SM_IMAGE_URI = os.environ.get(
    "SM_IMAGE_URI",
    f"763104351884.dkr.ecr.{REGION}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
)

LAGS = [1, 2, 3, 6, 12]
ENGINEERED_COLS = (
    [f"lag{l}" for l in LAGS]
    + ["roll3_mean", "roll6_mean", "roll6_std", "ewm3", "zero_ratio6",
       "months_since_nz", "month_sin", "month_cos", "quarter", "item_cv"]
)
FEATS = ENGINEERED_COLS + ["code"]


def K(p):
    return f"{ROOT}/{p}" if ROOT else p


def read_bytes(key):
    return boto3.client("s3", region_name=REGION).get_object(Bucket=BUCKET, Key=key)["Body"].read()


def put_bytes(key, b):
    boto3.client("s3", region_name=REGION).put_object(Bucket=BUCKET, Key=key, Body=b)


def exists(key):
    try:
        boto3.client("s3", region_name=REGION).head_object(Bucket=BUCKET, Key=key)
        return True
    except Exception:
        return False


def wape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    return float(np.abs(a - f).sum() / max(np.abs(a).sum(), 1e-9))


def psi(ref, cur, bins=5):
    ref, cur = np.asarray(ref, float), np.asarray(cur, float)
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3 or len(cur) == 0:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.clip(np.histogram(ref, edges)[0] / len(ref), 1e-4, None)
    c = np.clip(np.histogram(cur, edges)[0] / len(cur), 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))


def croston_sba_fc(y, h, alpha=0.15):
    """Croston's Syntetos-Boylan Approximation (SBA) for intermittent/lumpy demand."""
    y = np.asarray(y, float)
    nz_idx = np.flatnonzero(y > 0)
    if len(nz_idx) == 0:
        return np.zeros(h, dtype=float)
    if len(nz_idx) == 1:
        val = y[nz_idx[0]] / max(len(y), 1)
        return np.full(h, val, dtype=float)

    z = y[nz_idx[0]]
    p = float(nz_idx[0] + 1)
    q = 1
    for t in range(nz_idx[0] + 1, len(y)):
        if y[t] > 0:
            z = alpha * y[t] + (1.0 - alpha) * z
            p = alpha * q + (1.0 - alpha) * p
            q = 1
        else:
            q += 1

    rate = (1.0 - alpha / 2.0) * (z / max(p, 1.0))
    return np.full(h, max(rate, 0.0), dtype=float)


def ets_fc(y, h):
    """State-space Exponential Smoothing (ETS) with automatic seasonal/trend fallback."""
    y = np.asarray(y, float)
    try:
        if len(y) >= 24 and np.std(y) > 1e-6:
            res = ETSModel(y, error="add", trend="add", damped_trend=True, seasonal="add", seasonal_periods=12).fit(disp=False)
        elif len(y) >= 6 and np.std(y) > 1e-6:
            res = ETSModel(y, error="add", trend="add", damped_trend=True).fit(disp=False)
        else:
            return np.repeat(np.mean(y[-3:]), h)
        f = np.asarray(res.forecast(h), float)
        if not np.isfinite(f).all():
            raise ValueError("non-finite ETS forecast")
    except Exception:
        f = np.repeat(np.mean(y[-3:]), h)
    return np.clip(f, 0.0, None)


def stat_fc(model, y, h):
    if model == "croston":
        return croston_sba_fc(y, h)
    return ets_fc(y, h)


def _compute_features(d):
    """Computes lag, rolling, intermittency, and calendar features."""
    d = d.copy()
    g = d.groupby("item")["ys"]
    for l in LAGS:
        d[f"lag{l}"] = g.shift(l)

    shifted = g.shift(1)
    d["roll3_mean"] = shifted.groupby(d["item"]).transform(lambda s: s.rolling(3, min_periods=1).mean())
    d["roll6_mean"] = shifted.groupby(d["item"]).transform(lambda s: s.rolling(6, min_periods=1).mean())
    d["roll6_std"] = shifted.groupby(d["item"]).transform(lambda s: s.rolling(6, min_periods=2).std()).fillna(0.0)
    d["ewm3"] = shifted.groupby(d["item"]).transform(lambda s: s.ewm(span=3, adjust=False).mean())

    is_zero = (shifted == 0).astype(float)
    d["zero_ratio6"] = is_zero.groupby(d["item"]).transform(lambda s: s.rolling(6, min_periods=1).mean()).fillna(0.0)

    def _months_since_nz(arr):
        out = np.zeros(len(arr), dtype=float)
        cnt = 0.0
        for i, v in enumerate(arr):
            out[i] = cnt
            if np.isnan(v) or v > 0:
                cnt = 0.0
            else:
                cnt += 1.0
        return out

    d["months_since_nz"] = shifted.groupby(d["item"]).transform(
        lambda s: pd.Series(_months_since_nz(s.values), index=s.index)
    )

    m = d["date"].dt.month
    d["month_sin"] = np.sin(2.0 * np.pi * m / 12.0)
    d["month_cos"] = np.cos(2.0 * np.pi * m / 12.0)
    d["quarter"] = d["date"].dt.quarter
    return d


def xgb_fit_forecast(df, h):
    """Trains global XGBoost with strict training-window scaling (leak-free) and runs recursive forecast."""
    d = df.sort_values(["item", "date"]).reset_index(drop=True).copy()

    stats = d.groupby("item")["y"].agg(["mean", "std"]).fillna(0.0)
    scales = stats["mean"].clip(lower=1e-6)
    cv_map = (stats["std"] / scales).clip(0.0, 5.0).to_dict()
    d["ys"] = d["y"] / d["item"].map(scales)
    d["item_cv"] = d["item"].map(cv_map)

    codes = {k: i for i, k in enumerate(sorted(d["item"].unique()))}
    d["code"] = d["item"].map(codes)

    hist = _compute_features(d)
    train_mask = hist["lag1"].notna()

    model = xgb.XGBRegressor(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=2,
    )
    model.fit(hist.loc[train_mask, FEATS], hist.loc[train_mask, "ys"])

    base_hist = hist[["item", "date", "ys", "code", "item_cv"]].copy()
    for _ in range(h):
        nxt = base_hist.groupby("item", as_index=False)["date"].max()
        nxt["date"] = nxt["date"] + pd.DateOffset(months=1)
        nxt["ys"] = np.nan
        nxt["code"] = nxt["item"].map(codes)
        nxt["item_cv"] = nxt["item"].map(cv_map)
        base_hist = pd.concat([base_hist, nxt], ignore_index=True).sort_values(["item", "date"]).reset_index(drop=True)
        feat_df = _compute_features(base_hist)
        sel = feat_df["ys"].isna()
        preds = np.clip(model.predict(feat_df.loc[sel, FEATS]), 0.0, None)
        base_hist.loc[sel, "ys"] = preds

    out = {it: g["ys"].values[-h:] * scales[it] for it, g in base_hist.groupby("item")}
    meta = {
        "codes": codes,
        "scales": scales.to_dict(),
        "cv_map": cv_map,
    }
    return model, out, meta


def train_and_forecast(df, h=HORIZON, holdout=HOLDOUT):
    """Evaluates XGBoost, ETS, and Croston's SBA per item on holdout window and generates champion forecasts."""
    df = df.sort_values(["item", "date"]).reset_index(drop=True)
    cutoff = df["date"].max() - pd.DateOffset(months=holdout)
    tr, te = df[df["date"] <= cutoff].copy(), df[df["date"] > cutoff].copy()
    items = sorted(df["item"].unique())

    tr_clean = tr[["item", "date", "y"]].copy()
    _, xg_ho, _ = xgb_fit_forecast(tr_clean, holdout)

    abs_errors = {}
    actual_sums = {}
    for it in items:
        ytr = tr.loc[tr["item"] == it, "y"].values
        yte = te.loc[te["item"] == it, "y"].values
        if len(ytr) < MIN_TRAIN or len(yte) < holdout:
            continue
        actual_sums[it] = float(np.abs(yte).sum())
        abs_errors[it] = {
            "xgb": float(np.abs(yte - xg_ho[it]).sum()),
            "ets": float(np.abs(yte - stat_fc("ets", ytr, holdout)).sum()),
            "croston": float(np.abs(yte - stat_fc("croston", ytr, holdout)).sum()),
        }

    best = {it: min(err_map, key=err_map.get) for it, err_map in abs_errors.items()}
    total_best_err = sum(abs_errors[it][best[it]] for it in best)
    total_actual = sum(actual_sums.values())
    portfolio_wape = round(float(total_best_err / max(total_actual, 1e-9)), 4) if best else None

    model, xg_fc, meta = xgb_fit_forecast(df, h)

    sbc_map = df.groupby("item")["demand_class"].first().to_dict() if "demand_class" in df.columns else {}
    history_map = {it: g["y"].values.tolist() for it, g in df.groupby("item")}

    rows = []
    for it in items:
        g = df[df["item"] == it]
        y_hist = g["y"].values
        sbc = sbc_map.get(it, "intermittent" if np.mean(y_hist == 0) > 0.30 else "smooth")
        default_m = "croston" if sbc in ("intermittent", "lumpy") else "ets"
        m = best.get(it, default_m)
        f = xg_fc[it] if m == "xgb" else stat_fc(m, y_hist, h)
        dates = pd.date_range(g["date"].max() + pd.DateOffset(months=1), periods=h, freq="MS")
        rows.append(pd.DataFrame({"FCITEM": it, "demand_class": sbc, "date": dates, "forecast": np.round(f, 3), "model": m}))

    fc = pd.concat(rows, ignore_index=True)
    mix = {k: int(v) for k, v in pd.Series(list(best.values())).value_counts().items()} if best else {}
    sbc_mix = {k: int(v) for k, v in pd.Series(list(sbc_map.values())).value_counts().items()} if sbc_map else {}
    metrics = {
        "cv_wape": portfolio_wape,
        "model_mix": mix,
        "sbc_demand_classes": sbc_mix,
        "glue_features_used": all(col in df.columns for col in [f"lag{l}" for l in LAGS]),
        "candidate_models": ["xgb", "ets", "croston"],
        "n_items": len(items),
        "horizon": h,
    }
    artifact_meta = {**meta, "history": history_map}
    return fc, model, best, metrics, artifact_meta


def check_drift(df, bundle):
    ref = np.asarray(bundle["ref"])
    cur = np.log1p(df.loc[df["date"] > df["date"].max() - pd.DateOffset(months=3), "y"].values)
    p = psi(ref, cur)
    err, key = None, K("results/latest/forecast.csv")
    if exists(key):
        prev = pd.read_csv(io.BytesIO(read_bytes(key)), parse_dates=["date"])
        id_col = "FCITEM" if "FCITEM" in prev.columns else "item"
        j = prev.merge(df[["item", "date", "y"]], left_on=[id_col, "date"], right_on=["item", "date"])
        if len(j):
            err = wape(j["y"], j["forecast"])
    flags = []
    if p >= PSI_THRESHOLD:
        flags.append(f"data drift PSI={p:.3f} >= {PSI_THRESHOLD}")
    if err is not None and err >= ERR_THRESHOLD:
        flags.append(f"accuracy drift WAPE={err:.3f} >= {ERR_THRESHOLD}")
    return {
        "psi": round(p, 4),
        "wape_vs_actuals": None if err is None else round(err, 4),
        "drift": bool(flags),
        "reasons": flags,
    }


def publish_cloudwatch_metrics(metrics, drift):
    """Publishes custom forecasting & drift metrics to CloudWatch namespace FCITEM/Forecasting."""
    if ROOT:
        return  # Keep demo metrics isolated from prod dashboard
    try:
        cw = boto3.client("cloudwatch", region_name=REGION)
        mix = metrics.get("model_mix", {})
        data = [
            {"MetricName": "ActiveItemsCount", "Value": float(metrics.get("n_items", 0)), "Unit": "Count"},
            {"MetricName": "ChampionXGBCount", "Value": float(mix.get("xgb", 0)), "Unit": "Count"},
            {"MetricName": "ChampionETSCount", "Value": float(mix.get("ets", 0)), "Unit": "Count"},
            {"MetricName": "ChampionCrostonCount", "Value": float(mix.get("croston", 0)), "Unit": "Count"},
            {"MetricName": "DriftDetected", "Value": 1.0 if drift.get("drift") else 0.0, "Unit": "Count"},
        ]
        if metrics.get("cv_wape") is not None:
            data.append({"MetricName": "HoldoutWAPE", "Value": float(metrics["cv_wape"]), "Unit": "None"})
        if drift.get("psi") is not None:
            data.append({"MetricName": "DataDriftPSI", "Value": float(drift["psi"]), "Unit": "None"})
        if drift.get("wape_vs_actuals") is not None:
            data.append({"MetricName": "LiveWAPE", "Value": float(drift["wape_vs_actuals"]), "Unit": "None"})
        cw.put_metric_data(Namespace="FCITEM/Forecasting", MetricData=data)
    except Exception as e:
        print(f"Warning: CloudWatch metric publish failed: {e}", flush=True)


def register_and_deploy_sagemaker(bundle_bytes, metrics):
    """
    Packages model.tar.gz (using secure tempfile), registers version in SageMaker Model Registry,
    and deploys/updates the Real-Time Endpoint with Data Capture enabled for SageMaker Model Monitor.
    """
    if not ENABLE_SAGEMAKER or ROOT:
        return {"sagemaker_status": "skipped (demo or disabled)"}

    sm = boto3.client("sagemaker", region_name=REGION)
    sts = boto3.client("sts", region_name=REGION)
    acct = sts.get_caller_identity()["Account"]
    role_arn = SM_ROLE_ARN or f"arn:aws:iam::{acct}:role/forecast-pipeline-role"
    stamp = dt.datetime.utcnow().strftime("%Y%m%d-%H%M%S")

    # 1. Build model.tar.gz securely via tempfile.TemporaryDirectory (Bandit-clean)
    with tempfile.TemporaryDirectory() as tmpdir:
        bundle_path = os.path.join(tmpdir, "bundle.joblib")
        with open(bundle_path, "wb") as f:
            f.write(bundle_bytes)

        code_dir = os.path.join(tmpdir, "code")
        os.makedirs(code_dir, exist_ok=True)
        handler_src = os.path.join(os.path.dirname(os.path.abspath(__file__)), "sagemaker_handler.py")
        if os.path.exists(handler_src):
            with open(handler_src, "rb") as src, open(os.path.join(code_dir, "inference.py"), "wb") as dst:
                dst.write(src.read())
        with open(os.path.join(code_dir, "requirements.txt"), "w") as rf:
            rf.write("xgboost\nstatsmodels\njoblib\npandas\nnumpy\n")

        tar_path = os.path.join(tmpdir, "model.tar.gz")
        with tarfile.open(tar_path, "w:gz") as tar:
            tar.add(bundle_path, arcname="bundle.joblib")
            tar.add(code_dir, arcname="code")

        model_s3_key = f"sagemaker-models/{stamp}/model.tar.gz"
        with open(tar_path, "rb") as tf:
            put_bytes(model_s3_key, tf.read())

    model_data_url = f"s3://{BUCKET}/{model_s3_key}"

    # 2. Ensure SageMaker Model Package Group exists
    try:
        sm.describe_model_package_group(ModelPackageGroupName=SM_MODEL_GROUP)
    except Exception:
        sm.create_model_package_group(
            ModelPackageGroupName=SM_MODEL_GROUP,
            ModelPackageGroupDescription="FCITEM Multi-Model Demand Forecasting Registry (XGBoost / ETS / Croston)",
        )

    # 3. Save evaluation metrics JSON to S3 for Model Registry lineage
    eval_key = f"sagemaker-models/{stamp}/eval_metrics.json"
    eval_payload = {
        "regression_metrics": {
            "wape": {"value": float(metrics["cv_wape"] or 0.0), "standard_deviation": 0.0}
        }
    }
    put_bytes(eval_key, json.dumps(eval_payload).encode("utf-8"))

    cv_wape = metrics.get("cv_wape")
    approved = (cv_wape is not None) and (cv_wape <= APPROVAL_WAPE_THRESHOLD)
    approval_status = "Approved" if approved else "PendingManualApproval"

    pkg = sm.create_model_package(
        ModelPackageGroupName=SM_MODEL_GROUP,
        ModelPackageDescription=f"FCITEM champion ensemble | WAPE={cv_wape} | mix={metrics.get('model_mix')}",
        InferenceSpecification={
            "Containers": [
                {
                    "Image": SM_IMAGE_URI,
                    "ModelDataUrl": model_data_url,
                }
            ],
            "SupportedContentTypes": ["application/json"],
            "SupportedResponseMIMETypes": ["application/json"],
        },
        ModelMetrics={
            "ModelQuality": {
                "Statistics": {
                    "ContentType": "application/json",
                    "S3Uri": f"s3://{BUCKET}/{eval_key}",
                }
            }
        },
        ModelApprovalStatus=approval_status,
    )
    pkg_arn = pkg["ModelPackageArn"]
    print(f"Registered SageMaker ModelPackage: {pkg_arn} ({approval_status})", flush=True)

    if not approved:
        return {
            "model_package_arn": pkg_arn,
            "approval_status": approval_status,
            "endpoint_status": "not_deployed_pending_approval",
        }

    # 4. Create SageMaker Model + EndpointConfig (with Data Capture for Model Monitor) + Endpoint
    sm_model_name = f"fcitem-model-{stamp}"
    sm_cfg_name = f"fcitem-cfg-{stamp}"

    sm.create_model(
        ModelName=sm_model_name,
        PrimaryContainer={"ModelPackageName": pkg_arn},
        ExecutionRoleArn=role_arn,
    )
    sm.create_endpoint_config(
        EndpointConfigName=sm_cfg_name,
        ProductionVariants=[
            {
                "VariantName": "AllTraffic",
                "ModelName": sm_model_name,
                "InitialInstanceCount": 1,
                "InstanceType": SM_INSTANCE_TYPE,
                "InitialVariantWeight": 1.0,
            }
        ],
        DataCaptureConfig={
            "EnableCapture": True,
            "InitialSamplingPercentage": 100,
            "DestinationS3Uri": f"s3://{BUCKET}/sagemaker-datacapture",
            "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
            "CaptureContentTypeHeader": {"JsonContentTypes": ["application/json"]},
        },
    )

    try:
        ep = sm.describe_endpoint(EndpointName=SM_ENDPOINT_NAME)
        status = ep["EndpointStatus"]
        if status in ("Creating", "Updating"):
            ep_action = f"endpoint_busy_{status}"
        else:
            sm.update_endpoint(EndpointName=SM_ENDPOINT_NAME, EndpointConfigName=sm_cfg_name)
            ep_action = "updating_endpoint"
    except Exception:
        sm.create_endpoint(EndpointName=SM_ENDPOINT_NAME, EndpointConfigName=sm_cfg_name)
        ep_action = "creating_endpoint"

    return {
        "model_package_arn": pkg_arn,
        "approval_status": approval_status,
        "model_data_url": model_data_url,
        "endpoint_name": SM_ENDPOINT_NAME,
        "endpoint_action": ep_action,
    }


def notify(subject, msg):
    print(f"[NOTIFY] {subject}\n{msg}", flush=True)
    if SNS_TOPIC_ARN:
        boto3.client("sns", region_name=REGION).publish(TopicArn=SNS_TOPIC_ARN, Subject=subject[:100], Message=msg)


def main():
    df = pd.read_parquet(io.BytesIO(read_bytes(K("processed/clean.parquet"))))
    df["date"] = pd.to_datetime(df["date"])
    print(f"Loaded processed/clean.parquet | rows={len(df)} | FCITEMs={df['item'].nunique()}", flush=True)

    bkey = K("models/bundle.joblib")
    bundle = joblib.load(io.BytesIO(read_bytes(bkey))) if exists(bkey) else None

    drift = (
        check_drift(df, bundle)
        if bundle
        else {"drift": False, "reasons": ["no existing model"], "psi": None, "wape_vs_actuals": None}
    )
    today = dt.datetime.utcnow().strftime("%Y-%m-%d")
    put_bytes(K(f"results/drift/{today}_{MODE}.json"), json.dumps(drift).encode())
    print("DRIFT REPORT:", json.dumps(drift), flush=True)

    if bundle is None:
        reason = "initial training (no existing model)"
    elif MODE == "monthly":
        reason = "scheduled monthly retrain" + (f" | drift also detected: {drift['reasons']}" if drift["drift"] else "")
    elif drift["drift"]:
        reason = f"drift detected: {drift['reasons']}"
    else:
        publish_cloudwatch_metrics({"n_items": df["item"].nunique()}, drift)
        print("No drift detected - retraining skipped.", flush=True)
        return

    notify(
        "[Forecast] Model RE-TRAINING started",
        f"Reason: {reason}\nMode: {MODE}\nFCITEMs: {df['item'].nunique()} | Rows: {len(df)}\nDrift report: {json.dumps(drift)}",
    )

    fc, model, best, metrics, artifact_meta = train_and_forecast(df)
    ref = np.log1p(df.loc[df["date"] > df["date"].max() - pd.DateOffset(months=12), "y"].values)

    buf = io.BytesIO()
    joblib.dump(
        {
            "xgb": model,
            "best": best,
            "ref": ref,
            "trained_at": today,
            "last_date": str(df["date"].max().date()),
            **artifact_meta,
        },
        buf,
    )
    bundle_bytes = buf.getvalue()
    put_bytes(bkey, bundle_bytes)

    sm_info = register_and_deploy_sagemaker(bundle_bytes, metrics)

    csv_long = fc.to_csv(index=False).encode()
    put_bytes(K(f"results/run_date={today}/forecast.csv"), csv_long)
    put_bytes(K("results/latest/forecast.csv"), csv_long)

    wide = fc.pivot(index=["FCITEM", "demand_class", "model"], columns="date", values="forecast").reset_index()
    wide.columns = [c.strftime("%Y-%m") if hasattr(c, "strftime") else c for c in wide.columns]
    csv_wide = wide.to_csv(index=False).encode()
    put_bytes(K(f"results/run_date={today}/forecast_wide.csv"), csv_wide)
    put_bytes(K("results/latest/forecast_wide.csv"), csv_wide)

    metrics.update({"trained_at": today, "reason": reason, "drift": drift, "sagemaker": sm_info})
    mj = json.dumps(metrics, default=str).encode()
    put_bytes(K(f"results/run_date={today}/metrics.json"), mj)
    put_bytes(K("results/latest/metrics.json"), mj)

    publish_cloudwatch_metrics(metrics, drift)

    notify(
        "[Forecast] Model RE-TRAINING completed",
        f"Reason: {reason}\nHoldout WAPE: {metrics['cv_wape']}\nModel mix: {metrics['model_mix']}\n"
        f"SageMaker: {sm_info}\nResults: s3://{BUCKET}/{K('results/latest/')}",
    )


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        notify("[Forecast] Pipeline FAILED", repr(e))
        raise

In [ ]:
%%writefile app/api.py
import os
import io
import json
import time
import boto3
import pandas as pd
from typing import List
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel

BUCKET = os.environ.get("S3_BUCKET", "")
REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "eu-north-1"))
SM_ENDPOINT_NAME = os.environ.get("SM_ENDPOINT_NAME", "fcitem-forecast-endpoint")
CACHE_TTL_SEC = int(os.environ.get("CACHE_TTL_SEC", "300"))

app = FastAPI(
    title="FCITEM Live Demand Forecasting API",
    description="Live BI & Application Serving Layer backed by S3 Forecast Artifacts and SageMaker Real-Time Endpoint",
    version="1.0.0",
)

_CACHE = {"loaded_at": 0.0, "metrics": None, "forecast_df": None}


def _s3():
    return boto3.client("s3", region_name=REGION)


def _sm_runtime():
    return boto3.client("sagemaker-runtime", region_name=REGION)


def _load_latest(force: bool = False):
    now = time.time()
    if not force and _CACHE["forecast_df"] is not None and (now - _CACHE["loaded_at"]) < CACHE_TTL_SEC:
        return _CACHE["metrics"], _CACHE["forecast_df"]

    if not BUCKET:
        raise HTTPException(status_code=500, detail="S3_BUCKET environment variable is not configured")

    try:
        s3 = _s3()
        m_raw = s3.get_object(Bucket=BUCKET, Key="results/latest/metrics.json")["Body"].read()
        f_raw = s3.get_object(Bucket=BUCKET, Key="results/latest/forecast.csv")["Body"].read()
        metrics = json.loads(m_raw)
        df = pd.read_csv(io.BytesIO(f_raw))
        _CACHE["metrics"] = metrics
        _CACHE["forecast_df"] = df
        _CACHE["loaded_at"] = now
        return metrics, df
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Forecast artifacts unavailable in S3: {e}")


class BatchForecastRequest(BaseModel):
    fcitems: List[str]
    horizon: int = 18
    invoke_sagemaker: bool = True


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "bucket": BUCKET,
        "sagemaker_endpoint": SM_ENDPOINT_NAME,
        "cache_age_sec": round(time.time() - _CACHE["loaded_at"], 1) if _CACHE["loaded_at"] else None,
    }


@app.post("/reload")
def reload_cache():
    metrics, df = _load_latest(force=True)
    return {
        "status": "reloaded",
        "trained_at": metrics.get("trained_at"),
        "n_items": int(df["FCITEM"].nunique()),
    }


@app.get("/model-info")
def model_info():
    metrics, _ = _load_latest()
    return metrics


@app.get("/portfolio/summary")
def portfolio_summary(top_n: int = Query(10, ge=1, le=100)):
    metrics, df = _load_latest()
    totals = (
        df.groupby(["FCITEM", "demand_class", "model"], as_index=False)["forecast"]
        .sum()
        .rename(columns={"forecast": "total_18m_forecast"})
        .sort_values("total_18m_forecast", ascending=False)
    )
    return {
        "trained_at": metrics.get("trained_at"),
        "cv_wape": metrics.get("cv_wape"),
        "model_mix": metrics.get("model_mix"),
        "sbc_demand_classes": metrics.get("sbc_demand_classes"),
        "sagemaker": metrics.get("sagemaker"),
        "total_portfolio_forecast": round(float(totals["total_18m_forecast"].sum()), 2),
        "top_items": totals.head(top_n).to_dict(orient="records"),
    }


@app.get("/forecast/{fcitem}")
def get_item_forecast(
    fcitem: str,
    horizon: int = Query(18, ge=1, le=36),
    source: str = Query("auto", pattern="^(auto|batch|sagemaker)$"),
):
    if source in ("auto", "batch") and horizon == 18:
        metrics, df = _load_latest()
        sub = df[df["FCITEM"].astype(str) == str(fcitem)]
        if not sub.empty:
            return {
                "FCITEM": fcitem,
                "source": "batch_cache",
                "trained_at": metrics.get("trained_at"),
                "demand_class": sub["demand_class"].iloc[0],
                "model": sub["model"].iloc[0],
                "forecasts": sub[["date", "forecast"]].to_dict(orient="records"),
            }
        if source == "batch":
            raise HTTPException(status_code=404, detail=f"FCITEM '{fcitem}' not found in batch forecast")

    # Fallback or explicit live inference via SageMaker Real-Time Endpoint (feeds Data Capture & Model Monitor)
    try:
        resp = _sm_runtime().invoke_endpoint(
            EndpointName=SM_ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps({"fcitems": [fcitem], "horizon": horizon}),
        )
        payload = json.loads(resp["Body"].read().decode("utf-8"))
        return {"source": "sagemaker_realtime", **payload}
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"SageMaker endpoint invocation failed: {e}")


@app.post("/forecast/batch")
def batch_forecast(req: BatchForecastRequest):
    if not req.fcitems:
        raise HTTPException(status_code=400, detail="fcitems list cannot be empty")

    if req.invoke_sagemaker:
        try:
            resp = _sm_runtime().invoke_endpoint(
                EndpointName=SM_ENDPOINT_NAME,
                ContentType="application/json",
                Body=json.dumps({"fcitems": req.fcitems, "horizon": req.horizon}),
            )
            payload = json.loads(resp["Body"].read().decode("utf-8"))
            return {"source": "sagemaker_realtime", **payload}
        except Exception as e:
            raise HTTPException(status_code=502, detail=f"SageMaker batch invocation failed: {e}")

    metrics, df = _load_latest()
    sub = df[df["FCITEM"].astype(str).isin([str(x) for x in req.fcitems])]
    grouped = []
    for it, g in sub.groupby("FCITEM"):
        grouped.append({
            "FCITEM": it,
            "demand_class": g["demand_class"].iloc[0],
            "model": g["model"].iloc[0],
            "forecasts": g[["date", "forecast"]].head(req.horizon).to_dict(orient="records"),
        })
    return {"source": "batch_cache", "trained_at": metrics.get("trained_at"), "predictions": grouped}

In [ ]:
%%writefile app/tests/test_pipeline.py
import os, sys
import numpy as np, pandas as pd

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import pipeline as P
import sagemaker_handler as SMH


def make_df(n_items=4, n=36, seed=0):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2023-01-01", periods=n, freq="MS")
    rows = []
    for i in range(n_items):
        if i == n_items - 1:
            y = np.where(rng.random(n) < 0.35, rng.uniform(40, 120, n), 0.0)
        else:
            base = 100 * (i + 1)
            y = base * (1 + 0.3 * np.sin(2 * np.pi * np.arange(n) / 12)) + rng.normal(0, 0.05 * base, n) + np.arange(n) * i
        rows.append(pd.DataFrame({"item": f"FCITEM_{i:03d}", "date": dates, "y": np.clip(y, 0, None)}))
    return pd.concat(rows, ignore_index=True)


def test_forecast_shape_finite_nonnegative():
    df = make_df()
    fc, _, _, m, _ = P.train_and_forecast(df, h=6, holdout=3)
    assert len(fc) == df["item"].nunique() * 6
    assert "FCITEM" in fc.columns
    assert np.isfinite(fc["forecast"]).all() and (fc["forecast"] >= 0).all()
    assert set(fc["model"]) <= {"xgb", "ets", "croston"}
    assert m["candidate_models"] == ["xgb", "ets", "croston"]


def test_forecast_dates_follow_history():
    df = make_df()
    fc, _, _, _, _ = P.train_and_forecast(df, h=6, holdout=3)
    assert fc["date"].min() == df["date"].max() + pd.DateOffset(months=1)


def test_holdout_accuracy_reasonable():
    _, _, _, m, _ = P.train_and_forecast(make_df(), h=6, holdout=3)
    assert m["cv_wape"] is not None and m["cv_wape"] < 0.5


def test_croston_sba_intermittent_and_all_zero():
    sparse = np.array([0, 0, 50, 0, 0, 0, 60, 0, 0, 45, 0, 0], dtype=float)
    fc = P.croston_sba_fc(sparse, 4)
    assert len(fc) == 4 and np.all(fc > 0) and np.isfinite(fc).all()
    assert np.all(P.croston_sba_fc(np.zeros(12), 3) == 0.0)


def test_ets_fallback_on_short_series():
    f = P.ets_fc(np.array([10.0, 12.0, 14.0]), 3)
    assert len(f) == 3 and np.isfinite(f).all()


def test_psi_and_wape():
    rng = np.random.default_rng(1)
    a = rng.normal(5, 1, 500)
    assert P.psi(a, rng.normal(5, 1, 500)) < 0.1
    assert P.psi(a, rng.normal(8, 1, 500)) > 0.25
    assert P.wape([10, 10], [10, 10]) == 0
    assert abs(P.wape([10, 10], [5, 15]) - 0.5) < 1e-9


def test_sagemaker_handler_predict_fn():
    df = make_df(n_items=2, n=24)
    _, model, best, _, meta = P.train_and_forecast(df, h=3, holdout=3)
    bundle = {
        "xgb": model,
        "best": best,
        "trained_at": "2026-11-25",
        "last_date": str(df["date"].max().date()),
        **meta,
    }
    out = SMH.predict_fn({"fcitems": ["FCITEM_000", "FCITEM_001"], "horizon": 3}, bundle)
    assert len(out["predictions"]) == 2
    assert len(out["predictions"][0]["forecasts"]) == 3

In [ ]:
%%writefile app/tests/test_api.py
import os, sys
import pandas as pd
from fastapi.testclient import TestClient

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import api


def test_health_endpoint():
    client = TestClient(api.app)
    resp = client.get("/health")
    assert resp.status_code == 200
    assert resp.json()["status"] == "healthy"


def test_cached_forecast_and_portfolio_endpoints(monkeypatch):
    sample_metrics = {
        "trained_at": "2026-11-25",
        "cv_wape": 0.142,
        "model_mix": {"xgb": 1, "croston": 1},
        "sbc_demand_classes": {"smooth": 1, "intermittent": 1},
    }
    sample_df = pd.DataFrame([
        {"FCITEM": "ITEM_A", "demand_class": "smooth", "date": "2026-01-01", "forecast": 120.5, "model": "xgb"},
        {"FCITEM": "ITEM_B", "demand_class": "intermittent", "date": "2026-01-01", "forecast": 15.0, "model": "croston"},
    ])
    monkeypatch.setattr(api, "_load_latest", lambda force=False: (sample_metrics, sample_df))

    client = TestClient(api.app)
    r1 = client.get("/portfolio/summary")
    assert r1.status_code == 200
    assert r1.json()["total_portfolio_forecast"] == 135.5

    r2 = client.get("/forecast/ITEM_A?source=batch")
    assert r2.status_code == 200
    assert r2.json()["model"] == "xgb"

## C. Pre-Docker Quality & Security Verification Gate (`pytest` + `bandit`)
Auto-installs test/security dependencies (`pytest`, `bandit`, `fastapi`, `xgboost`, `statsmodels`, `httpx`) into the active kernel Python (`sys.executable`) if missing, then runs all unit tests, FastAPI endpoint tests, and Bandit SAST checks **before** packaging.

In [ ]:
open("app/requirements.txt", "w").write("""xgboost
statsmodels
pandas
numpy
pyarrow
joblib
boto3
fastapi
uvicorn[standard]
pydantic
httpx
pytest
""")
open("app/Dockerfile", "w").write("""FROM python:3.10-slim
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY pipeline.py api.py sagemaker_handler.py ./
COPY tests ./tests
RUN python -m pytest -q tests
EXPOSE 8080
CMD ["python", "pipeline.py"]
""")

# Ensure pytest, bandit & app dependencies are installed in the active notebook kernel
print("=== Ensuring Test & Security Dependencies in Kernel Environment ===")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", "app/requirements.txt", "bandit"],
    check=True,
)

# Run pre-Docker unit & API tests using sys.executable
print("=== Running Pre-Docker Pytest Suite ===")
res = subprocess.run([sys.executable, "-m", "pytest", "-v", "app/tests"], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Pre-Docker unit/API tests failed! Fix errors before packaging.")

# Run Bandit SAST static security check
b_res = subprocess.run(
    [sys.executable, "-m", "bandit", "-r", "app", "-ll", "-x", "app/tests"],
    capture_output=True, text=True
)
print(b_res.stdout)
if b_res.returncode != 0:
    print(b_res.stderr)
    raise RuntimeError("Bandit SAST security check flagged issues! Fix before packaging.")
print("=== Bandit SAST Security Check PASSED ===")

## D. Package `app.zip` & Push Docker Image to ECR (No-CodeBuild Workflow)
Pre-Docker unit/API tests (`pytest`) and security checks (`bandit`) already passed in Section C.
1. Run the cell below to package `app.zip` (note: `Dockerfile` also executes `pytest` during `docker build`).
2. Upload `app.zip` to AWS CloudShell (or run locally if Docker is installed) and execute the printed commands to build & push to ECR.

In [ ]:
try:
    ecr.create_repository(repositoryName=ECR_REPO, imageScanningConfiguration={"scanOnPush": True})
    print("Created ECR repository:", ECR_REPO)
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("ECR repository exists:", ECR_REPO)

shutil.make_archive("app", "zip", "app")
print("app.zip generated! Upload app.zip to AWS CloudShell and run:")
print(f"""
mkdir -p fcitem_build && unzip -o app.zip -d fcitem_build && cd fcitem_build
aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com
docker build -t {ECR_URI} .
docker push {ECR_URI}
""")

## E. Production Infrastructure: ECS Batch + ECS Live BI Service + Step Functions + CloudWatch Dashboard
*Run Cells E1–E4 after the Docker image is in ECR (`forecast-pipeline:latest`).*

In [ ]:
# Cell E1: SNS Alerts + ECS Cluster + Batch Task Definition + Live BI FastAPI Service (Port 8080)
topic_arn = sns.create_topic(Name=TOPIC_NAME)["TopicArn"]
sns.subscribe(TopicArn=topic_arn, Protocol="email", Endpoint=ALERT_EMAIL)
print("-> IMPORTANT: Confirm SNS subscription in inbox:", ALERT_EMAIL)

ecs.create_cluster(clusterName=CLUSTER)
for lg in [LOG_GROUP, "/ecs/forecast-live-service"]:
    try:
        logs.create_log_group(logGroupName=lg)
    except logs.exceptions.ResourceAlreadyExistsException:
        pass

# 1. Batch Training & Drift Task Definition
td_batch = ecs.register_task_definition(
    family=TASK_FAMILY,
    networkMode="awsvpc",
    requiresCompatibilities=["FARGATE"],
    cpu="2048",
    memory="8192",
    executionRoleArn=PIPELINE_ROLE_ARN,
    taskRoleArn=PIPELINE_ROLE_ARN,
    containerDefinitions=[
        {
            "name": CONTAINER,
            "image": ECR_URI,
            "essential": True,
            "environment": [
                {"name": "S3_BUCKET", "value": BUCKET},
                {"name": "AWS_REGION", "value": REGION},
                {"name": "SNS_TOPIC_ARN", "value": topic_arn},
                {"name": "HORIZON", "value": str(HORIZON)},
                {"name": "MODE", "value": "monthly"},
                {"name": "ENABLE_SAGEMAKER", "value": "true"},
                {"name": "SM_MODEL_GROUP", "value": SM_MODEL_GROUP},
                {"name": "SM_ENDPOINT_NAME", "value": SM_ENDPOINT_NAME},
                {"name": "SM_INSTANCE_TYPE", "value": SM_INSTANCE_TYPE},
                {"name": "SM_ROLE_ARN", "value": PIPELINE_ROLE_ARN},
            ],
            "logConfiguration": {
                "logDriver": "awslogs",
                "options": {
                    "awslogs-group": LOG_GROUP,
                    "awslogs-region": REGION,
                    "awslogs-stream-prefix": "ecs",
                },
            },
        }
    ],
)
print("Registered Batch Task Definition:", td_batch["taskDefinition"]["taskDefinitionArn"])

# 2. Live BI Endpoint Task Definition (FastAPI + Uvicorn on Port 8080)
LIVE_FAMILY = "forecast-live-api-task"
td_live = ecs.register_task_definition(
    family=LIVE_FAMILY,
    networkMode="awsvpc",
    requiresCompatibilities=["FARGATE"],
    cpu="1024",
    memory="2048",
    executionRoleArn=PIPELINE_ROLE_ARN,
    taskRoleArn=PIPELINE_ROLE_ARN,
    containerDefinitions=[
        {
            "name": CONTAINER,
            "image": ECR_URI,
            "essential": True,
            "command": ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8080"],
            "portMappings": [{"containerPort": 8080, "protocol": "tcp"}],
            "environment": [
                {"name": "S3_BUCKET", "value": BUCKET},
                {"name": "AWS_REGION", "value": REGION},
                {"name": "SM_ENDPOINT_NAME", "value": SM_ENDPOINT_NAME},
            ],
            "logConfiguration": {
                "logDriver": "awslogs",
                "options": {
                    "awslogs-group": "/ecs/forecast-live-service",
                    "awslogs-region": REGION,
                    "awslogs-stream-prefix": "api",
                },
            },
        }
    ],
)

# 3. Create or update persistent ECS Fargate Service for BI Dashboards
net_cfg = {
    "awsvpcConfiguration": {
        "subnets": subnets,
        "securityGroups": [sg_id],
        "assignPublicIp": "ENABLED",
    }
}
existing = ecs.describe_services(cluster=CLUSTER, services=[LIVE_SERVICE])["services"]
active_svcs = [s for s in existing if s["status"] == "ACTIVE"]
if active_svcs:
    ecs.update_service(
        cluster=CLUSTER,
        service=LIVE_SERVICE,
        taskDefinition=LIVE_FAMILY,
        desiredCount=1,
        networkConfiguration=net_cfg,
        forceNewDeployment=True,
    )
    print("Updated persistent ECS Live Service:", LIVE_SERVICE)
else:
    ecs.create_service(
        cluster=CLUSTER,
        serviceName=LIVE_SERVICE,
        taskDefinition=LIVE_FAMILY,
        desiredCount=1,
        launchType="FARGATE",
        networkConfiguration=net_cfg,
    )
    print("Created persistent ECS Live Service:", LIVE_SERVICE)

In [ ]:
# Cell E2: Retrieve Live ECS BI Endpoint URL & Attach Inbuilt SageMaker Model Monitor
def get_live_ecs_api_url():
    tasks = ecs.list_tasks(cluster=CLUSTER, serviceName=LIVE_SERVICE, desiredStatus="RUNNING")["taskArns"]
    if not tasks:
        print("Live ECS task still provisioning... re-run this helper in ~30s.")
        return None
    desc = ecs.describe_tasks(cluster=CLUSTER, tasks=tasks)["tasks"][0]
    eni_id = None
    for att in desc.get("attachments", []):
        for d in att.get("details", []):
            if d["name"] == "networkInterfaceId":
                eni_id = d["value"]
    if not eni_id:
        return None
    eni = ec2.describe_network_interfaces(NetworkInterfaceIds=[eni_id])["NetworkInterfaces"][0]
    pub_ip = eni.get("Association", {}).get("PublicIp")
    if pub_ip:
        url = f"http://{pub_ip}:8080"
        print(f"Live BI API URL: {url}")
        print(f"  Swagger Docs : {url}/docs")
        print(f"  Health Check : {url}/health")
        print(f"  Portfolio BI : {url}/portfolio/summary")
        return url
    return None

def attach_sagemaker_model_monitor():
    """Attaches an hourly inbuilt SageMaker Model Monitor Data Quality schedule to fcitem-forecast-endpoint."""
    try:
        ep = sm.describe_endpoint(EndpointName=SM_ENDPOINT_NAME)
    except Exception:
        print("SageMaker endpoint not created yet. Run the monthly training pipeline first.")
        return
    if ep["EndpointStatus"] != "InService":
        print(f"SageMaker endpoint status is {ep['EndpointStatus']}. Wait until InService, then re-run.")
        return

    job_def_name = "fcitem-dq-monitor-jobdef"
    schedule_name = "fcitem-data-quality-monitor"
    monitor_image = f"895015795356.dkr.ecr.{REGION}.amazonaws.com/sagemaker-model-monitor-analyzer"

    try:
        sm.create_data_quality_job_definition(
            JobDefinitionName=job_def_name,
            DataQualityAppSpecification={"ImageUri": monitor_image},
            DataQualityJobInput={
                "EndpointInput": {
                    "EndpointName": SM_ENDPOINT_NAME,
                    "LocalPath": "/opt/ml/processing/input/endpoint",
                }
            },
            DataQualityJobOutputConfig={
                "MonitoringOutputs": [
                    {
                        "S3Output": {
                            "S3Uri": f"s3://{BUCKET}/sagemaker-monitor-reports",
                            "LocalPath": "/opt/ml/processing/output",
                            "S3UploadMode": "EndOfJob",
                        }
                    }
                ]
            },
            JobResources={
                "ClusterConfig": {
                    "InstanceCount": 1,
                    "InstanceType": "ml.m5.large",
                    "VolumeSizeInGB": 20,
                }
            },
            RoleArn=PIPELINE_ROLE_ARN,
        )
        print("Created SageMaker DataQualityJobDefinition:", job_def_name)
    except Exception as e:
        print("DataQualityJobDefinition status:", e)

    try:
        sm.create_monitoring_schedule(
            MonitoringScheduleName=schedule_name,
            MonitoringScheduleConfig={
                "ScheduleConfig": {"ScheduleExpression": "cron(0 * ? * * *)"},
                "MonitoringJobDefinitionName": job_def_name,
                "MonitoringType": "DataQuality",
            },
        )
        print("Attached inbuilt SageMaker Model Monitor schedule:", schedule_name)
    except Exception as e:
        print("MonitoringSchedule status:", e)

get_live_ecs_api_url()

In [ ]:
# Cell E3: Step Functions State Machine (Glue -> ECS) + EventBridge Cron Rules
cluster_arn = f"arn:aws:ecs:{REGION}:{ACCOUNT_ID}:cluster/{CLUSTER}"
fail = [{"ErrorEquals": ["States.ALL"], "Next": "NotifyFailure"}]

definition = {
    "Comment": "Hardened Glue FCITEM prep -> ECS Fargate train, SageMaker deploy & drift pipeline",
    "StartAt": "Preprocess",
    "States": {
        "Preprocess": {
            "Type": "Task",
            "Resource": "arn:aws:states:::glue:startJobRun.sync",
            "Parameters": {
                "JobName": GLUE_JOB,
                "Arguments": {
                    "--S3_BUCKET": BUCKET,
                    "--RAW_PREFIX": RAW_PREFIX,
                    "--PROCESSED_KEY": "processed/clean.parquet",
                    "--DQ_KEY": "processed/dq_report.json",
                    "--ITEM_COL": ITEM_COL,
                },
            },
            "Retry": [
                {
                    "ErrorEquals": [
                        "Glue.ConcurrentRunsExceededException",
                        "Glue.AWSGlueException",
                        "States.TaskFailed",
                    ],
                    "IntervalSeconds": 30,
                    "MaxAttempts": 2,
                    "BackoffRate": 2.0,
                }
            ],
            "Catch": fail,
            "Next": "TrainForecast",
        },
        "TrainForecast": {
            "Type": "Task",
            "Resource": "arn:aws:states:::ecs:runTask.sync",
            "Parameters": {
                "Cluster": cluster_arn,
                "TaskDefinition": TASK_FAMILY,
                "LaunchType": "FARGATE",
                "NetworkConfiguration": {
                    "AwsvpcConfiguration": {
                        "Subnets": subnets,
                        "SecurityGroups": [sg_id],
                        "AssignPublicIp": "ENABLED",
                    }
                },
                "Overrides": {
                    "ContainerOverrides": [
                        {
                            "Name": CONTAINER,
                            "Environment": [{"Name": "MODE", "Value.$": "$.MODE"}],
                        }
                    ]
                },
            },
            "Catch": fail,
            "End": True,
        },
        "NotifyFailure": {
            "Type": "Task",
            "Resource": "arn:aws:states:::sns:publish",
            "Parameters": {
                "TopicArn": topic_arn,
                "Subject": "Forecast pipeline FAILED",
                "Message.$": "States.JsonToString($)",
            },
            "Next": "Failed",
        },
        "Failed": {"Type": "Fail"},
    },
}

try:
    sm_arn = sfn.create_state_machine(
        name=SFN_NAME, definition=json.dumps(definition), roleArn=PIPELINE_ROLE_ARN
    )["stateMachineArn"]
except sfn.exceptions.StateMachineAlreadyExists:
    sm_arn = f"arn:aws:states:{REGION}:{ACCOUNT_ID}:stateMachine:{SFN_NAME}"
    sfn.update_state_machine(
        stateMachineArn=sm_arn, definition=json.dumps(definition), roleArn=PIPELINE_ROLE_ARN
    )

for name, cron, mode in [
    ("forecast-monthly-retrain", "cron(0 6 1 * ? *)", "monthly"),      # 1st of month 06:00 UTC
    ("forecast-weekly-drift-check", "cron(0 6 ? * MON *)", "drift"),   # Every Monday 06:00 UTC
]:
    events.put_rule(Name=name, ScheduleExpression=cron, State="ENABLED")
    events.put_targets(
        Rule=name,
        Targets=[
            {
                "Id": "sfn",
                "Arn": sm_arn,
                "RoleArn": PIPELINE_ROLE_ARN,
                "Input": json.dumps({"MODE": mode}),
            }
        ],
    )

print("State Machine ARN:", sm_arn)
print("EventBridge schedules active: forecast-monthly-retrain & forecast-weekly-drift-check")

In [ ]:
# Cell E4: Create Unified CloudWatch Dashboard (FCITEM-MLOps-Dashboard)
dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Forecast Accuracy (Holdout WAPE & Live WAPE)",
                "region": REGION,
                "stat": "Average",
                "period": 3600,
                "metrics": [
                    ["FCITEM/Forecasting", "HoldoutWAPE"],
                    ["FCITEM/Forecasting", "LiveWAPE"],
                ],
            },
        },
        {
            "type": "metric",
            "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Data Drift PSI & Drift Trigger Flag",
                "region": REGION,
                "stat": "Maximum",
                "period": 3600,
                "metrics": [
                    ["FCITEM/Forecasting", "DataDriftPSI"],
                    ["FCITEM/Forecasting", "DriftDetected"],
                ],
            },
        },
        {
            "type": "metric",
            "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "title": "Champion Model Split (XGBoost vs ETS vs Croston)",
                "region": REGION,
                "stat": "Maximum",
                "period": 3600,
                "metrics": [
                    ["FCITEM/Forecasting", "ChampionXGBCount"],
                    ["FCITEM/Forecasting", "ChampionETSCount"],
                    ["FCITEM/Forecasting", "ChampionCrostonCount"],
                    ["FCITEM/Forecasting", "ActiveItemsCount"],
                ],
            },
        },
        {
            "type": "metric",
            "x": 12, "y": 6, "width": 12, "height": 6,
            "properties": {
                "title": "Glue Data Quality (Input Rows, Valid SKUs, Excluded SKUs)",
                "region": REGION,
                "stat": "Maximum",
                "period": 3600,
                "metrics": [
                    ["FCITEM/Forecasting", "GlueInputRows"],
                    ["FCITEM/Forecasting", "GlueValidItems"],
                    ["FCITEM/Forecasting", "GlueExcludedItems"],
                ],
            },
        },
        {
            "type": "metric",
            "x": 0, "y": 12, "width": 12, "height": 6,
            "properties": {
                "title": "ECS Live BI Service (CPU & Memory Utilization)",
                "region": REGION,
                "stat": "Average",
                "period": 300,
                "metrics": [
                    ["AWS/ECS", "CPUUtilization", "ClusterName", CLUSTER, "ServiceName", LIVE_SERVICE],
                    ["AWS/ECS", "MemoryUtilization", "ClusterName", CLUSTER, "ServiceName", LIVE_SERVICE],
                ],
            },
        },
        {
            "type": "metric",
            "x": 12, "y": 12, "width": 12, "height": 6,
            "properties": {
                "title": "SageMaker Real-Time Endpoint (Invocations, Latency & Errors)",
                "region": REGION,
                "stat": "Sum",
                "period": 300,
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", SM_ENDPOINT_NAME, "VariantName", "AllTraffic"],
                    ["AWS/SageMaker", "Invocation4XXErrors", "EndpointName", SM_ENDPOINT_NAME, "VariantName", "AllTraffic"],
                    ["AWS/SageMaker", "Invocation5XXErrors", "EndpointName", SM_ENDPOINT_NAME, "VariantName", "AllTraffic"],
                ],
            },
        },
    ]
}

cw.put_dashboard(DashboardName=DASHBOARD_NAME, DashboardBody=json.dumps(dashboard_body))
print(f"CloudWatch Dashboard ready: https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#dashboards:name={DASHBOARD_NAME}")

In [ ]:
# Cell E5 (Optional): Trigger real end-to-end execution (Glue -> ECS -> SageMaker Registry & Endpoint)
# ex = sfn.start_execution(stateMachineArn=sm_arn, input=json.dumps({"MODE": "monthly"}))
# print("Started Step Functions Execution:", ex["executionArn"])
# Once SageMaker endpoint reaches InService (~6-8 mins), attach inbuilt Model Monitor:
# attach_sagemaker_model_monitor()

## F. Live Demo Suite (Isolated under `s3://<bucket>/demo/` — Real FCITEM Dataset & Prod SageMaker Endpoint Untouched)
1. **Demo 1 (`monthly`):** Baseline 36 months (`FCITEM_000` .. `FCITEM_019`) -> Initial training + SNS email + S3 forecasts.
2. **Demo 2 (`drift`):** Normal Month 37 arrives -> PSI & WAPE within thresholds -> Retraining skipped.
3. **Demo 3 (`drift`):** Month 38 experiences a **6x demand shock** -> Drift detected -> Auto-retrains & sends alert email.

In [ ]:
DEMO_ROOT = "demo"

def demo_series(n_total=38, n_items=20, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2023-01-01", periods=n_total, freq="MS")
    rows = []
    for i in range(n_items):
        if i % 4 == 3:
            y = np.where(rng.random(n_total) < 0.35, rng.uniform(80, 250, n_total), 0.0)
            sbc = "intermittent"
        else:
            base = 50 + 30 * i
            y = (
                base * (1 + 0.25 * np.sin(2 * np.pi * np.arange(n_total) / 12))
                + rng.normal(0, 0.06 * base, n_total)
                + np.arange(n_total) * 0.5
            )
            sbc = "smooth"
        rows.append(pd.DataFrame({"item": f"FCITEM_{i:03d}", "date": dates, "y": np.clip(y, 0, None), "demand_class": sbc}))
    return pd.concat(rows, ignore_index=True), dates

full, dates = demo_series()
shocked = full.copy()
shocked.loc[shocked["date"] == dates[-1], "y"] *= 6.0

def put_processed(d):
    b = io.BytesIO()
    d.to_parquet(b, index=False)
    s3.put_object(Bucket=BUCKET, Key=f"{DEMO_ROOT}/processed/clean.parquet", Body=b.getvalue())

def run_ecs(mode):
    t0 = time.time()
    r = ecs.run_task(
        cluster=CLUSTER,
        taskDefinition=TASK_FAMILY,
        launchType="FARGATE",
        networkConfiguration={
            "awsvpcConfiguration": {
                "subnets": subnets,
                "securityGroups": [sg_id],
                "assignPublicIp": "ENABLED",
            }
        },
        overrides={
            "containerOverrides": [
                {
                    "name": CONTAINER,
                    "environment": [
                        {"name": "MODE", "value": mode},
                        {"name": "ROOT", "value": DEMO_ROOT},
                    ],
                }
            ]
        },
    )
    if r.get("failures"):
        print("Task launch failures:", r["failures"])
        return
    arn = r["tasks"][0]["taskArn"]
    task_id = arn.split("/")[-1]
    print(f"Fargate task started: {task_id} (mode={mode}) | waiting...")
    ecs.get_waiter("tasks_stopped").wait(
        cluster=CLUSTER, tasks=[arn], WaiterConfig={"Delay": 10, "MaxAttempts": 90}
    )
    t = ecs.describe_tasks(cluster=CLUSTER, tasks=[arn])["tasks"][0]
    print(f"Exit code: {t['containers'][0].get('exitCode')} | Elapsed: {time.time() - t0:.1f}s")
    time.sleep(5)
    for e in logs.get_log_events(
        logGroupName=LOG_GROUP, logStreamName=f"ecs/{CONTAINER}/{task_id}", startFromHead=True
    )["events"]:
        print(e["message"].rstrip())

def show_latest():
    m = json.loads(s3.get_object(Bucket=BUCKET, Key=f"{DEMO_ROOT}/results/latest/metrics.json")["Body"].read())
    print("--- Latest Metrics & Model Mix ---")
    print(json.dumps(m, indent=2))
    display(pd.read_csv(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=f"{DEMO_ROOT}/results/latest/forecast_wide.csv")["Body"].read())).head(8))

In [ ]:
# DEMO 1: Baseline 36 months -> Initial training + SNS email + S3 forecasts
put_processed(full[full["date"] <= dates[35]])
run_ecs("monthly")
show_latest()

In [ ]:
# DEMO 2: Normal Month 37 arrives -> Drift check says OK, retraining skipped
put_processed(full[full["date"] <= dates[36]])
run_ecs("drift")

In [ ]:
# DEMO 3: Month 38 with 6x demand shock -> Drift detected -> Auto-retrain + SNS email
put_processed(shocked[shocked["date"] <= dates[37]])
run_ecs("drift")
show_latest()